In [4]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark=SparkSession.builder\
        .appName('Spark_Skew_Demo')\
        .getOrCreate()
# =============================================
#  构造 100 条测试数据（真实热点比例）
# =============================================
data=[]
# 1. 热点 key = 1001 → 50条数据（50%，严重热点）
for i in range(50):
    data.append((1001,'hot_data'))
# 2. 普通 key 10个，每个5条 → 共50条
normal_keys=[1002,1003,1004,1005,1006,1007,1008,1009,1010,1011]
for key in normal_keys:
    for i in range(5):
        data.append((key,'normal_data'))
df = spark.createDataFrame(data, ["join_key", "value"])
print("===== 总数据量：", df.count(), "条 =====")
df.show()



===== 总数据量： 100 条 =====
+--------+--------+
|join_key|   value|
+--------+--------+
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
+--------+--------+
only showing top 20 rows



In [2]:
# 第 1 步：统计热点 key 占比（判断是否热点）
#统计每个key数量
df_hot_stat=df.groupBy('join_key').count().orderBy(desc('count'))
df_hot_stat.show(10)

+--------+-----+
|join_key|count|
+--------+-----+
|    1001|   50|
|    1002|    5|
|    1003|    5|
|    1005|    5|
|    1004|    5|
|    1006|    5|
|    1007|    5|
|    1008|    5|
|    1009|    5|
|    1010|    5|
+--------+-----+
only showing top 10 rows



In [3]:
# 计算每个key的占比
total=df.count()
df_hot_stat=df_hot_stat.withColumn('precent',round(col('count')/total *100,2))
df_hot_stat.show(15,truncate=False)

+--------+-----+-------+
|join_key|count|precent|
+--------+-----+-------+
|1001    |50   |50.0   |
|1002    |5    |5.0    |
|1003    |5    |5.0    |
|1005    |5    |5.0    |
|1004    |5    |5.0    |
|1006    |5    |5.0    |
|1007    |5    |5.0    |
|1008    |5    |5.0    |
|1009    |5    |5.0    |
|1010    |5    |5.0    |
|1011    |5    |5.0    |
+--------+-----+-------+



In [4]:
# 第 2 步：热点分离（把 热点 key 和 非热点 key 拆开）
# 热点数据（单独抽出来）
df_hot=df.filter(col('join_key')==1001)
# 正常数据（无热点）
df_normal=df.filter(col('join_key')!=1001)
print("热点数据条数：", df_hot.count())
df_hot.show()
print("正常数据条数：", df_normal.count())
df_normal.show()


热点数据条数： 50
+--------+--------+
|join_key|   value|
+--------+--------+
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
|    1001|hot_data|
+--------+--------+
only showing top 20 rows

正常数据条数： 50
+--------+-----------+
|join_key|      value|
+--------+-----------+
|    1002|normal_data|
|    1002|normal_data|
|    1002|normal_data|
|    1002|normal_data|
|    1002|normal_data|
|    1003|normal_data|
|    1003|normal_data|
|    1003|normal_data|
|    1003|normal_data|
|    1003|normal_data|
|    1004|normal_data|
|    1004|normal_data|
|    1004|normal_data|
|    1004|normal_data|
|    1004|normal_data|
|    1005|normal_data|
|    1005|normal_data|
|    1005|no

In [5]:
# 第 3 步：分别 JOIN（演示真实业务场景）
# 构造一个小表 dim，模拟大小表 join
dim_data=[
          (1001,'热点用户'),
          (1002,'普通用户'),
          (1003,'普通用户')
          ]
for k in normal_keys:
    dim_data.append((k,'普通用户'))
df_dim=spark.createDataFrame(dim_data,['join_key','user_type'])
df_dim.show(20)

+--------+---------+
|join_key|user_type|
+--------+---------+
|    1001| 热点用户|
|    1002| 普通用户|
|    1003| 普通用户|
|    1002| 普通用户|
|    1003| 普通用户|
|    1004| 普通用户|
|    1005| 普通用户|
|    1006| 普通用户|
|    1007| 普通用户|
|    1008| 普通用户|
|    1009| 普通用户|
|    1010| 普通用户|
|    1011| 普通用户|
+--------+---------+



In [6]:
# --------------------------
# 1. 热点数据：广播JOIN（最优）
# --------------------------
df_hot_join=df_hot.join(broadcast(df_dim),'join_key')
df_hot_join.show(50)

+--------+--------+---------+
|join_key|   value|user_type|
+--------+--------+---------+
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|


In [7]:
# --------------------------
# 2. 正常数据：普通JOIN
# --------------------------
df_normal_join=df_normal.join(df_dim,'join_key')
df_normal_join.show(80)

+--------+-----------+---------+
|join_key|      value|user_type|
+--------+-----------+---------+
|    1002|normal_data| 普通用户|
|    1002|normal_data| 普通用户|
|    1002|normal_data| 普通用户|
|    1002|normal_data| 普通用户|
|    1002|normal_data| 普通用户|
|    1002|normal_data| 普通用户|
|    1002|normal_data| 普通用户|
|    1002|normal_data| 普通用户|
|    1002|normal_data| 普通用户|
|    1002|normal_data| 普通用户|
|    1003|normal_data| 普通用户|
|    1003|normal_data| 普通用户|
|    1003|normal_data| 普通用户|
|    1003|normal_data| 普通用户|
|    1003|normal_data| 普通用户|
|    1003|normal_data| 普通用户|
|    1003|normal_data| 普通用户|
|    1003|normal_data| 普通用户|
|    1003|normal_data| 普通用户|
|    1003|normal_data| 普通用户|
|    1004|normal_data| 普通用户|
|    1004|normal_data| 普通用户|
|    1004|normal_data| 普通用户|
|    1004|normal_data| 普通用户|
|    1004|normal_data| 普通用户|
|    1005|normal_data| 普通用户|
|    1005|normal_data| 普通用户|
|    1005|normal_data| 普通用户|
|    1005|normal_data| 普通用户|
|    1005|normal_data| 普通用户|
|    1006|normal_data| 普通用户|
| 

In [8]:
# 第 4 步：最后聚合 union 合并结果
# 合并热点 + 正常数据
df_final=df_hot_join.union(df_normal_join)
print("===== 最终聚合后总条数：", df_final.count(), "=====")
df_final.show()


===== 最终聚合后总条数： 110 =====
+--------+--------+---------+
|join_key|   value|user_type|
+--------+--------+---------+
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
+--------+--------+---------+
only showing top 20 rows



In [10]:
df_final.groupBy('join_key','user_type').count().orderBy(desc('count'))
print(df_final.count())
df_final.show()

110
+--------+--------+---------+
|join_key|   value|user_type|
+--------+--------+---------+
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
|    1001|hot_data| 热点用户|
+--------+--------+---------+
only showing top 20 rows



In [5]:
spark.stop()